# Imports

Loads libraries, resolves paths, and initialises the result containers.

- `PREPROC_DIR` points to the preprocessed `.pickle` files produced by `preprocessing.ipynb`.
- `SAVE_DIR` (`results/`) is where the fitted TRF bundles are written.
- `TMIN` / `TMAX` set the lag window (−0.8 – 0.8 s) used for both forward and backward models.
- `_make_results()` builds the nested dict that accumulates per-subject TRF objects, keyed by direction (`fw` / `bw`), sensor array (`scalp` / `ceegrid`), attention condition (`attended` / `ignored` / `null`), and regressor (`env` / `onset`).

Run this cell before any other cell in the notebook.

In [ ]:
from pathlib import Path
import eelbrain as eel
import numpy as np
import re

ROOT = Path.cwd()
SAVE_DIR = ROOT / 'results'
SAVE_DIR.mkdir(exist_ok=True)
PREPROC_DIR = ROOT / 'data' / 'derivatives' / 'preprocessed'

TMIN = -0.8
TMAX = 0.8

subjects = [
    p.name.removeprefix("sub-")
    for p in PREPROC_DIR.iterdir()
    if p.is_dir() and p.name.startswith("sub-")
]

def _make_results():
    regressors = ['env', 'onset']
    conditions = ['attended', 'ignored', 'null']
    acq_types  = ['scalp', 'ceegrid']
    return {
        'models': {
            'fw': {acq: {
                cond: {reg: [] for reg in regressors}
                for cond in conditions
            } for acq in acq_types},
            'bw': {
                acq: {
                    cond: {reg: [] for reg in regressors}
                    for cond in conditions
                }
                for acq in acq_types
            }
        }
    }

sustA = _make_results()
switA = _make_results()
convA = _make_results()

# Run TRFs

Fits forward and backward TRF models for every subject, condition, sensor array, attention condition, and regressor combination, then saves the results to disk.

For each preprocessed `.pickle` file the cell:
1. Loads the z-scored EEG epochs and predictor NDVars (attended, ignored, null/shifted).
2. Fits a **backward model** (`pred → eeg`, lag −0.8 – 0.8 s) — estimates how well the acoustic predictor decodes the EEG; the cross-validated correlation *r* quantifies decoding accuracy.
3. Fits a **forward model** (`eeg → pred`, same lag window) — estimates the temporal response function (TRF) as a sensor × time map.
4. Both models use L1 error, a 50 ms basis, and leave-one-trial-out cross-validation (`partitions = n_trials`).

Results are accumulated into the `sustA`, `switA`, and `convA` dicts and saved as:
- `results/sustA_models.pkl`
- `results/switA_models.pkl`
- `results/convA_models.pkl`

In [ ]:
for s, subject in enumerate(subjects):
    print(f"Subject {s+1}/{len(subjects)}")

    EEG_DIR = PREPROC_DIR / f"sub-{subject}"

    files = [f for f in EEG_DIR.rglob("*") if f.is_file()]
    print(f"Found {len(files)} files for subject nr {s+1}.")

    for file in files:
        df = eel.load.unpickle(file)
        acq = re.search(r'_acq-([^_.]+)', file.name).group(1)
        acq_type = re.match(r'^(scalp|ceegrid)', acq).group(1)
        task_cap = re.sub(r'^(scalp|ceegrid)', '', acq)
        task = task_cap[0].lower() + task_cap[1:]

        eeg = df['eeg']
        predictors = df['predictors']
        nPartitions = eeg.shape[0]

        for attention in predictors.keys():
            for regressor in predictors[attention].keys():
                print(f"    Running: Subject {subject} --- {task} --- {acq_type} --- {attention} --- {regressor}")

                pred = predictors[attention][regressor]
                trf_bw = eel.boosting(pred, eeg, TMIN, TMAX, basis=0.05, error='l1', partitions=nPartitions, test=1, partition_results=True)
                trf_fw = eel.boosting(eeg, pred, TMIN, TMAX, basis=0.05, error='l1', partitions=nPartitions, test=1, partition_results=True)

                if task == 'sustA':
                    sustA['models']['bw'][acq_type][attention][regressor].append(trf_bw)
                    sustA['models']['fw'][acq_type][attention][regressor].append(trf_fw)
                elif task == 'switA':
                    switA['models']['bw'][acq_type][attention][regressor].append(trf_bw)
                    switA['models']['fw'][acq_type][attention][regressor].append(trf_fw)
                elif task == 'convA':   
                    convA['models']['bw'][acq_type][attention][regressor].append(trf_bw)
                    convA['models']['fw'][acq_type][attention][regressor].append(trf_fw)


fname = SAVE_DIR / 'sustA_models.pkl'
eel.save.pickle(sustA, fname)

fname = SAVE_DIR / 'switA_models.pkl'
eel.save.pickle(switA, fname)

fname = SAVE_DIR / 'convA_models.pkl'
eel.save.pickle(convA, fname)
        

# Optimal lag analysis

Sweeps a short boosting window across a range of lags to find the time offset at which the EEG and the acoustic predictor are maximally correlated. This identifies the optimal integration window without assuming a fixed lag a priori.

**Method:**
- A sliding window of 45 ms is stepped in 15 ms increments from −0.6 s to +0.6 s.
- At each lag position a TRF model is fitted (`eelbrain.boosting`, L1 error, no cross-validation) and the resulting correlation *r* is recorded.
- The per-lag correlations are assembled into an NDVar over the lag-time axis and saved to `results/tmp/` so individual lags can be re-loaded without re-fitting.

**Output NDVars:**
- Backward (`pred → eeg`): shape `(n_lags,)` — one scalar *r* per lag.
- Forward (`eeg → pred`): shape `(n_sensors, n_lags)` — per-sensor *r* per lag.

Final results are saved as:
- `results/sustA_optLag.pkl`
- `results/switA_optLag.pkl`
- `results/convA_optLag.pkl`

These files are consumed by `plot_optLag()` in `plotting.py`.

In [ ]:
TMP_DIR = ROOT / 'results' / 'tmp'
TMP_DIR.mkdir(exist_ok=True)

sustA = _make_results()
switA = _make_results()
convA = _make_results()

lag_range = (-0.6, 0.6)
window = 0.045
overlap = 0.03
tstep = window - overlap
lags = np.arange(lag_range[0], lag_range[1], tstep)

time = []
for i, lag in enumerate(lags):
    time.append(lag+window/2)
uts = eel.UTS(time[0], tstep, len(time))

stims = ['attended_files', 'ignored_files']


for s, subject in enumerate(subjects):
    print(f"Subject {s+1}/{len(subjects)}")

    EEG_DIR = PREPROC_DIR / f"sub-{subject}"

    files = [f for f in EEG_DIR.rglob("*") if f.is_file()]
    print(f"Found {len(files)} files for subject nr {s+1}.")

    for file in files:
        df = eel.load.unpickle(file)
        acq = re.search(r'_acq-([^_.]+)', file.name).group(1)
        acq_type = re.match(r'^(scalp|ceegrid)', acq).group(1)
        task_cap = re.sub(r'^(scalp|ceegrid)', '', acq)
        task = task_cap[0].lower() + task_cap[1:]


        eeg = df['eeg']
        predictors = df['predictors']
        nPartitions = eeg.shape[0]

        for attention in predictors.keys():
            for regressor in predictors[attention].keys():
                pred = predictors[attention][regressor]

                filename_bw = f'{subject}_{task}_{acq_type}_{regressor}_bw_lagsCorr.pickle'
                dst_bw = TMP_DIR / filename_bw

                if not dst_bw.exists():
                    bw_r = []
                    for i, lag in enumerate(lags):
                        tmin = lag
                        tmax = lag + window

                        if not dst_bw.exists():
                            decoder = eel.boosting(pred, eeg,
                                    tmin, tmax,
                                    error='l1',
                                    basis=0.05,
                                    partitions=nPartitions,
                                    test=1
                                )
                            bw_r.append(decoder.r)
                    name = f'Correlation {subject} {task} {acq_type} {regressor} BW'
                    ndvar_bw = eel.NDVar(bw_r, uts, name=name)
                    eel.save.pickle(ndvar_bw, dst_bw)
                else:
                    ndvar_bw = eel.load.unpickle(dst_bw)
                
                filename_fw = f'{subject}_{task}_{acq_type}_{regressor}_fw_lagsCorr.pickle'
                dst_fw = TMP_DIR / filename_fw

                if not dst_fw.exists():
                    fw_r = []
                    for i, lag in enumerate(lags):
                        tmin = lag
                        tmax = lag + window

                        if not dst_fw.exists():
                            encoder = eel.boosting(eeg, pred,
                                    tmin, tmax,
                                    error='l1',
                                    basis=0.05,
                                    partitions=nPartitions,
                                    test=1
                                )
                            fw_r.append(encoder.r)

                    sensor_dim = fw_r[0].sensor
                    data_fw = np.stack([ndvar.x for ndvar in fw_r])
                    data_fw = data_fw.T
                    name = f'Correlation {subject} {task} {acq_type} {regressor} FW'
                    ndvar_fw = eel.NDVar(data_fw, (sensor_dim, uts), name=name)
                    eel.save.pickle(ndvar_fw, dst_fw)
                else:
                    ndvar_fw = eel.load.unpickle(dst_fw)

                
                if task == 'sustA':
                    sustA['models']['bw'][acq_type][attention][regressor].append(ndvar_bw)
                    sustA['models']['fw'][acq_type][attention][regressor].append(ndvar_fw)
                elif task == 'switA':
                    switA['models']['bw'][acq_type][attention][regressor].append(ndvar_bw)
                    switA['models']['fw'][acq_type][attention][regressor].append(ndvar_fw)
                elif task == 'convA':
                    convA['models']['bw'][acq_type][attention][regressor].append(ndvar_bw)
                    convA['models']['fw'][acq_type][attention][regressor].append(ndvar_fw)


fname = SAVE_DIR / 'sustA_optLag.pkl'
eel.save.pickle(sustA, fname)

fname = SAVE_DIR / 'switA_optLag.pkl'
eel.save.pickle(switA, fname)

fname = SAVE_DIR / 'convA_optLag.pkl'
eel.save.pickle(convA, fname)
